<a href="https://colab.research.google.com/github/kashaf11303/urdu-ocr-codesaviours-si26-kashaf/blob/main/SI26_Week4_kashaf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opencv-python-headless pillow
import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
import cv2
import os

def preprocess_image(image_path, save_path):
    img = cv2.imread(image_path)

    if img is None:
        print(f'Could not load: {image_path}')
        return
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (512, 128))
    denoised = cv2.fastNlMeansDenoising(resized, h=10)
    _, binary = cv2.threshold(
        denoised,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    cv2.imwrite(save_path, binary)

    return binary

In [4]:
import os

input_folder = "/content/drive/MyDrive/200 images"
output_folder = "/content/drive/MyDrive/processed_images"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        image_path = os.path.join(input_folder, filename)
        save_path = os.path.join(output_folder, filename)

        preprocess_image(image_path, save_path)

print("✅ All images have been preprocessed successfully!")

✅ All images have been preprocessed successfully!


In [5]:
!pip install transformers torch pillow pandas

In [6]:
import torch
from torch.utils.data import Dataset
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

In [7]:
!pip install sentencepiece tiktoken

In [22]:
!pip install -U transformers sentencepiece tokenizers tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 48.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [8]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-handwritten"
)

print("Processor loaded successfully!")

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [4]:
csv_path = "/content/drive/MyDrive/labels_fixed.csv"

In [5]:
import pandas as pd
import os

df = pd.read_csv(csv_path)

df["image"] = df["image"].apply(
    lambda x: "/content/drive/MyDrive/200 images/" + os.path.basename(x)
)

df.to_csv("/content/drive/MyDrive/labels_fixed.csv", index=False)

print("labels_fixed.csv created successfully")

labels_fixed.csv created successfully


In [6]:
!pip install transformers torch pillow pandas

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd


class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        # Load and convert image
        image = Image.open(row['image']).convert("RGB")

        # Process image for the model
        encoding = self.processor(
            image,
            return_tensors="pt"
        )
        pixel_values = encoding.pixel_values.squeeze()

        # Process the text label
        labels = self.processor.tokenizer(
            row['text'],
            padding="max_length",
            max_length=128
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [24]:
!pip uninstall -y transformers tokenizers sentencepiece
!pip install transformers==4.41.2 tokenizers==0.19.1 sentencepiece==0.1.99

Found existing installation: transformers 5.14.1
Uninstalling transformers-5.14.1:
  Successfully uninstalled transformers-5.14.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: sentencepiece 0.2.2
Uninstalling sentencepiece-0.2.2:
  Successfully uninstalled sentencepiece-0.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 36.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.3 MB/s eta 0:00:00
  Created wheel for sentencepiece: filename=sentencepiece-0.1.99-cp312-cp312-linux_x86_64.whl size=1266901 sha256=b20f456b08f1e694ae261f17797b7d34237aebcd1ccbaabb7c544db850c1d6bd
 

In [1]:
import transformers
import tokenizers
import sentencepiece

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("SentencePiece:", sentencepiece.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Transformers: 4.41.2
Tokenizers: 0.19.1
SentencePiece: 0.1.99


In [21]:
!pip list | grep -E "transformers|tokenizers|sentencepiece"

sentence-transformers                 5.6.0
sentencepiece                         0.2.2
tokenizers                            0.22.2
transformers                          5.14.1


In [2]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-handwritten"
)

print("Processor loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Processor loaded successfully!


In [7]:
# Load the TrOCR processor
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-handwritten",
    use_fast=False
)

# Create dataset
dataset = UrduOCRDataset(csv_path, processor)

# Test it loads correctly
sample = dataset[0]

print("Sample pixel_values shape:", sample["pixel_values"].shape)
print("Sample labels shape:", sample["labels"].shape)
print("Dataset is working correctly!")

# Create train / test split (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print(f"Training samples: {train_size}")
print(f"Testing samples: {test_size}")

Dataset loaded: 200 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 160
Testing samples: 40


In [8]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

if device == 'cpu':
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > GPU')

# Load the pretrained model
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
model = VisionEncoderDecoderModel.from_pretrained(
    'microsoft/trocr-base-printed'
)
model = model.to(device)

# Configure model for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cpu
Go to Runtime > Change runtime type > GPU


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


In [9]:
train_dataset
test_dataset
model

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_fea

In [10]:
from torch.utils.data import DataLoader
from transformers import AdamW

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=4)

# Optimiser
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training batches per epoch: 40
Ready to train!
